# TSFM evaluation analysis

Read synchronized evaluation metrics, marginal tables, win rates, and plots without loading a forecasting model.

## Environment

In [ ]:
from pathlib import Path
import sys

on_drive = False

if on_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    project_root = Path('/content/drive/MyDrive/Recherche/Thèse Gaspard/Codes/tsfm_evaluation')
    data_path = Path('/content/drive/MyDrive/Recherche/Thèse Gaspard/Datasets')
else:
    project_root = Path.cwd().resolve()
    if project_root.name == 'src':
        project_root = project_root.parent
    data_path = project_root / 'datasets'

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
%cd $project_root

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import Image, display, clear_output

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)

## Select a report

The report is generated by the Slurm workflow. `TABLE_METRIC` chooses any error or inference-time marginal table metric; every error plot still contains all five error metrics.

In [ ]:
family = 'univariate'
mode = 'full'
report_metric = 'nmse'

run_root = project_root / 'outputs' / family
report_dir = project_root / 'outputs' / 'reports' / family / mode
results = pd.read_csv(report_dir / 'results.csv')
plot_index = pd.read_csv(report_dir / 'plot_index.csv')
win_rates = pd.read_csv(report_dir / 'chronos_win_rates.csv')
by_dataset = pd.read_csv(report_dir / f'{report_metric}_average_by_dataset.csv')
by_setting = pd.read_csv(report_dir / f'{report_metric}_average_by_setting.csv')

display(by_dataset)
display(by_setting)

## Dataset and setting explorer

In [ ]:
datasets = sorted(results['dataset'].unique())
dataset_widget = widgets.Dropdown(options=datasets, description='Dataset:')
setting_widget = widgets.Dropdown(description='L:H:')
metric_widget = widgets.Dropdown(options=['mse', 'mae', 'nmse', 'nmae', 'mase'], value='nmse', description='Metric:')
output = widgets.Output()

def update_settings(*_):
    selected = results[results['dataset'] == dataset_widget.value]
    settings = sorted({f"{int(row.lags)}:{int(row.horizon)}" for row in selected.itertuples()})
    setting_widget.options = settings

def draw(*_):
    if not setting_widget.value:
        return
    lags, horizon = map(int, setting_widget.value.split(':'))
    metric = metric_widget.value
    selected = results[(results['dataset'] == dataset_widget.value) & (results['lags'] == lags) & (results['horizon'] == horizon)]
    columns = ['model', f'metrics_{metric}', f'metrics_sample_std_{metric}', f'metrics_user_mean_{metric}', f'metrics_user_std_{metric}', f'metrics_w10_{metric}', 'inference_seconds', 'inference_seconds_per_user', 'inference_seconds_per_series_window']
    with output:
        clear_output(wait=True)
        display(selected[[column for column in columns if column in selected]])
        wins = win_rates[(win_rates['dataset'] == dataset_widget.value) & (win_rates['lags'] == lags) & (win_rates['horizon'] == horizon) & (win_rates['metric'] == metric)]
        if not wins.empty:
            display(wins)
        indexed = plot_index[(plot_index['dataset'] == dataset_widget.value) & (plot_index['setting'] == f'{lags}_{horizon}') & (plot_index['metric'] == metric)]
        for row in indexed.itertuples():
            print(row.kind)
            display(Image(filename=str(report_dir / row.png)))

dataset_widget.observe(update_settings, names='value')
dataset_widget.observe(draw, names='value')
setting_widget.observe(draw, names='value')
metric_widget.observe(draw, names='value')
update_settings()
draw()
display(widgets.HBox([dataset_widget, setting_widget, metric_widget]), output)

## Inspect the underlying lightweight artifacts

In [ ]:
row = results.iloc[0]
run_dir = run_root / row['run_path']
print(run_dir)
display(pd.read_csv(run_dir / 'per_user_metrics.csv').head())
display(pd.read_csv(run_dir / 'window_metrics.csv').head())
display(pd.read_csv(run_dir / 'horizon_metrics.csv').head())